In [ ]:
# 1. Встановлення бібліотек
!pip install earthengine-api geemap --quiet

In [ ]:
# 2: Підключення до Google Earth Engine
import ee
import geemap
# Автентифікація - підключаємо ваш Google акаунт до GEE
ee.Authenticate()

In [ ]:
# 3: Ініціалізація GEE з вашим проектом
ee.Initialize(project='curious-nucleus-494617-f5')

print("Google Earth Engine підключено успішно!")

In [ ]:
from ee.filter import Filter
# Крок 4: Завантажуємо межі Житомирської області
# FAO GAUL - це офіційна база меж регіонів від ООН
zhytomyr = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq('ADM1_NAME', "Zhytomyrs'ka"))

print("Житомирська область знайдена!")
print("Кількість об'єктів:", zhytomyr.size().getInfo())

In [ ]:
# Шукаємо всі регіони України щоб побачити точну назву
ukraine = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq('ADM0_NAME', 'Ukraine'))
names = ukraine.aggregate_array('ADM1_NAME').getInfo()
print("Всі області України в базі:")
for name in names:
  print(name)


In [ ]:
Map = geemap.Map()
Map.centerObject(zhytomyr, zoom=8)
Map.addLayer(zhytomyr, {'color': 'orange'}, 'Житомирська область')
Map



In [ ]:
# 6: Завантажуємо знімки Sentinel-2 за 2021 рік (до війни)
# Беремо квітень-червень - це час максимального NDVI пшениці

sentinel = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(zhytomyr).filterDate('2021-04-01', '2021-06-30').filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
print("Знімки знайдено!")
print("Кількість знімків:", sentinel.size().getInfo())


In [ ]:
# 7: Рахуємо NDVI для кожного знімка
def calculate_ndvi(image):
    # NIR = інфрачервоний канал (B8)
    # RED = червоний канал (B4)
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)
# Застосовуємо до всіх 56 знімків
sentinel_ndvi = sentinel.map(calculate_ndvi)
# Беремо середнє значення по всіх знімках
ndvi_mean = sentinel_ndvi.select('NDVI').mean().clip(zhytomyr)
print("NDVI розраховано для 56 знімків!")

In [ ]:
# 8: Показуємо карту NDVI на екрані
# Кольорова шкала: червоний = низький NDVI, зелений = високий NDVI
ndvi_params = {'min': 0, "max": 0.8, 'palette': ['red', 'yellow', 'lightgreen', 'green', 'darkgreen']}
Map2 = geemap.Map()
Map2.centerObject(zhytomyr, zoom=8)
Map2.addLayer(ndvi_mean, ndvi_params, 'NDVI Житомирська 2021')
Map2.addLayer(zhytomyr, {'color': 'gray'}, 'Межа області')

Map2

In [ ]:
# 9: Завантажуємо знімки для трьох років
def get_ndvi_mean(year):
    collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(zhytomyr).filterDate(f'{year}-04-01', f'{year}-06-30').filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)).map(calculate_ndvi) \
        .select('NDVI') \
        .mean() \
        .clip(zhytomyr)
    return collection

ndvi_2021 = get_ndvi_mean(2021)
ndvi_2022 = get_ndvi_mean(2022)
ndvi_2024 = get_ndvi_mean(2024)

print("NDVI розраховано для 2021, 2022, 2024!")

In [ ]:
# 10: Показуємо три карти для порівняння
ndvi_params = {'min': 0, 'max': 0.8, 'palette': ['red', 'yellow', 'lightgreen', 'green', 'darkgreen']}
# Карта 2021 (до війни)
Map_2021 = geemap.Map()
Map_2021.centerObject(zhytomyr, zoom=8)
Map_2021.addLayer(ndvi_2021, ndvi_params, 'NDVI 2021 (до війни)')
Map_2021.addLayer(zhytomyr, {'color': 'gray'}, 'Межа')
print("Карта 2021 (до війни):")
display(Map_2021)
# Карта 2022 (перший рік війни)
Map_2022 = geemap.Map()
Map_2022.centerObject(zhytomyr, zoom=8)
Map_2022.addLayer(ndvi_2022, ndvi_params, 'NDVI 2022 (початок війни)')
Map_2022.addLayer(zhytomyr, {'color': 'gray'}, 'Межа')
print("Карта 2022 (початок війни):")
display(Map_2022)
# Карта 2024
Map_2024 = geemap.Map()
Map_2024.centerObject(zhytomyr, zoom=8)
Map_2024.addLayer(ndvi_2024, ndvi_params, 'NDVI 2024')
Map_2024.addLayer(zhytomyr, {'color': 'gray'}, 'Межа')
print("Карта 2024:")
display(Map_2024)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 10: Вивантажуємо дані NDVI як масиви
region = zhytomyr.geometry()

# Параметри вивантаження
scale = 1000  # 1км на піксель - достатньо для області

ndvi_2021_arr = geemap.ee_to_numpy(ndvi_2021, region=region, scale=scale)
ndvi_2022_arr = geemap.ee_to_numpy(ndvi_2022, region=region, scale=scale)
ndvi_2024_arr = geemap.ee_to_numpy(ndvi_2024, region=region, scale=scale)

print("Дані вивантажено!")
print("Розмір масиву:", ndvi_2021_arr.shape)

In [ ]:
# 11: Малюємо три карти поряд
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

роки = [ndvi_2021_arr, ndvi_2022_arr, ndvi_2024_arr]
назви = ['2021 (до військових дій)', '2022 (початок військових дій)', '2024 (продовження військових дій)']
кольори = 'RdYlGn'  # червоний-жовтий-зелений

for i, (дані, назва) in enumerate(zip(роки, назви)):
    # Прибираємо нулі (вода, хмари)
    дані_clean = np.where(дані[:,:,0] == 0, np.nan, дані[:,:,0])

    im = axes[i].imshow(дані_clean,
                         cmap=кольори,
                         vmin=0, vmax=0.8,
                         interpolation='bilinear')
    axes[i].set_title(назва, fontsize=14, fontweight='bold', pad=10)
    axes[i].axis('off')  # прибираємо осі
    plt.colorbar(im, ax=axes[i],
                 label='NDVI',
                 fraction=0.046)

fig.suptitle('Динаміка NDVI пшениці — Житомирська область\nКвітень–Червень 2021, 2022, 2024',
             fontsize=16, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('ndvi_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Рисунок збережено як ndvi_comparison.png!")

2019: NDVI = 0.61
2020: NDVI = 0.58
2021: NDVI = 0.63
2022: NDVI = 0.41  ← падіння через війну
2023: NDVI = 0.52
2024: NDVI = 0.55

In [ ]:
# 12: Рахуємо середнє NDVI по всій області для кожного року
def get_ndvi_mean_value(year):
    ndvi_image = get_ndvi_mean(year)
    mean_val = ndvi_image.reduceRegion(reducer=ee.Reducer.mean(), geometry=zhytomyr.geometry(), scale=1000, maxPixels=1e9).get('NDVI').getInfo()
    return round(mean_val, 4)

# Рахуємо для всіх років
роки = [2019, 2020, 2021, 2022, 2023, 2024]
значення = []

for рік in роки:
    val = get_ndvi_mean_value(рік)
    значення.append(val)
    print(f"{рік}: NDVI = {val}")

print("\n Статистика по роках готова!")

In [ ]:
kherson = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq('ADM1_NAME', "Khersons'ka"))

print("✅ Херсонська область знайдена!")
print("Кількість об'єктів:", kherson.size().getInfo())

In [ ]:
Map = geemap.Map()
Map.centerObject(kherson, zoom=8)
Map.addLayer(kherson, {'color': 'gray'}, 'Херсонська область')
Map

In [ ]:
#  Завантажуємо знімки Sentinel-2 за 2021 рік (до війни)
# Беремо квітень-червень - це час максимального NDVI пшениці

sentinel = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(kherson).filterDate('2021-04-01', '2021-06-30').filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
print("Знімки знайдено!")
print("Кількість знімків:", sentinel.size().getInfo())

In [ ]:
# 7: Рахуємо NDVI для кожного знімка
def calculate_ndvi(image):
    # NIR = інфрачервоний канал (B8)
    # RED = червоний канал (B4)
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)
# Застосовуємо до всіх 111 знімків
sentinel_ndvi = sentinel.map(calculate_ndvi)
# Беремо середнє значення по всіх знімках
ndvi_mean = sentinel_ndvi.select('NDVI').mean().clip(kherson)
print("NDVI розраховано для 111 знімків!")

In [ ]:
# Показуємо карту NDVI на екрані
# Кольорова шкала: червоний = низький NDVI, зелений = високий NDVI
ndvi_params = {'min': 0, "max": 0.8, 'palette': ['red', 'yellow', 'lightgreen', 'green', 'darkgreen']}
Map2 = geemap.Map()
Map2.centerObject(kherson, zoom=8)
Map2.addLayer(ndvi_mean, ndvi_params, 'NDVI {Херсонська}} 2021')
Map2.addLayer(kherson, {'color': 'orange'}, 'Межа області')

Map2

In [ ]:
# 9: Завантажуємо знімки для трьох років
def get_ndvi_mean(year):
    collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(kherson).filterDate(f'{year}-04-01', f'{year}-06-30').filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)).map(calculate_ndvi) \
        .select('NDVI') \
        .mean() \
        .clip(kherson)
    return collection

ndvi_2021 = get_ndvi_mean(2021)
ndvi_2022 = get_ndvi_mean(2022)
ndvi_2024 = get_ndvi_mean(2024)

print("NDVI розраховано для 2021, 2022, 2024!")

In [ ]:
# 10: Показуємо три карти для порівняння
ndvi_params = {'min': 0, 'max': 0.8, 'palette': ['red', 'yellow', 'lightgreen', 'green', 'darkgreen']}
# Карта 2021 (до війни)
Map_2021 = geemap.Map()
Map_2021.centerObject(kherson, zoom=8)
Map_2021.addLayer(ndvi_2021, ndvi_params, 'NDVI 2021 (до війни)')
Map_2021.addLayer(kherson, {'color': 'orange'}, 'Межа')
print("Карта 2021 (до війни):")
display(Map_2021)
# Карта 2022 (перший рік війни)
Map_2022 = geemap.Map()
Map_2022.centerObject(kherson, zoom=8)
Map_2022.addLayer(ndvi_2022, ndvi_params, 'NDVI 2022 (початок війни)')
Map_2022.addLayer(kherson, {'color': 'orange'}, 'Межа')
print("Карта 2022 (початок війни):")
display(Map_2022)
# Карта 2024
Map_2024 = geemap.Map()
Map_2024.centerObject(kherson, zoom=8)
Map_2024.addLayer(ndvi_2024, ndvi_params, 'NDVI 2024')
Map_2024.addLayer(kherson, {'color': 'orange'}, 'Межа')
print("Карта 2024:")
display(Map_2024)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 10: Вивантажуємо дані NDVI як масиви
region = kherson.geometry()

# Параметри вивантаження
scale = 1000  # 1км на піксель - достатньо для області

ndvi_2021_arr = geemap.ee_to_numpy(ndvi_2021, region=region, scale=scale)
ndvi_2022_arr = geemap.ee_to_numpy(ndvi_2022, region=region, scale=scale)
ndvi_2024_arr = geemap.ee_to_numpy(ndvi_2024, region=region, scale=scale)

print("Дані вивантажено!")
print("Розмір масиву:", ndvi_2021_arr.shape)

In [ ]:
# 11: Малюємо три карти поряд
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

роки = [ndvi_2021_arr, ndvi_2022_arr, ndvi_2024_arr]
назви = ['2021 (до військових дій)', '2022 (початок військових дій)', '2024 (продовження військових дій)']
кольори = 'RdYlGn'  # червоний-жовтий-зелений

for i, (дані, назва) in enumerate(zip(роки, назви)):
    # Прибираємо нулі (вода, хмари)
    дані_clean = np.where(дані[:,:,0] == 0, np.nan, дані[:,:,0])

    im = axes[i].imshow(дані_clean,
                         cmap=кольори,
                         vmin=0, vmax=0.8,
                         interpolation='bilinear')
    axes[i].set_title(назва, fontsize=14, fontweight='bold', pad=10)
    axes[i].axis('off')  # прибираємо осі
    plt.colorbar(im, ax=axes[i],
                 label='NDVI',
                 fraction=0.046)

fig.suptitle('Динаміка NDVI пшениці — Херсонська область\nКвітень–Червень 2021, 2022, 2024',
             fontsize=16, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('ndvi_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Рисунок збережено як ndvi_comparison.png!")

In [ ]:
# 12: Рахуємо середнє NDVI по всій області для кожного року
def get_ndvi_mean_value(year):
    ndvi_image = get_ndvi_mean(year)
    mean_val = ndvi_image.reduceRegion(reducer=ee.Reducer.mean(), geometry=kherson.geometry(), scale=1000, maxPixels=1e9).get('NDVI').getInfo()
    return round(mean_val, 4)

# Рахуємо для всіх років
роки = [2019, 2020, 2021, 2022, 2023, 2024]
значення = []

for рік in роки:
    val = get_ndvi_mean_value(рік)
    значення.append(val)
    print(f"{рік}: NDVI = {val}")

print("\n Статистика по роках готова!")

"Динаміка вегетаційного індексу NDVI сільськогосподарських культур Житомирської та Херсонської областей в умовах воєнного стану (2019–2024)"

In [ ]:
# 14: Порівняльний графік двох областей
fig, ax = plt.subplots(figsize=(12, 6))

роки = [2019, 2020, 2021, 2022, 2023, 2024]

zhytomyr_ndvi = [0.5424, 0.4579, 0.5318, 0.5610, 0.5809, 0.6183]
kherson_ndvi  = [0.3974, 0.3933, 0.4144, 0.3898, 0.4336, 0.3622]

# Лінії областей
ax.plot(роки, zhytomyr_ndvi,
        color='darkgreen',
        linewidth=2.5,
        marker='o',
        markersize=8,
        label='Житомирська обл.')

ax.plot(роки, kherson_ndvi,
        color='darkorange',
        linewidth=2.5,
        marker='s',
        markersize=8,
        label='Херсонська обл.')

# Підписи значень
for рік, val in zip(роки, zhytomyr_ndvi):
    ax.annotate(f'{val}',
                xy=(рік, val),
                xytext=(0, 10),
                textcoords='offset points',
                ha='center', fontsize=9,
                color='darkgreen')

for рік, val in zip(роки, kherson_ndvi):
    ax.annotate(f'{val}',
                xy=(рік, val),
                xytext=(0, -16),
                textcoords='offset points',
                ha='center', fontsize=9,
                color='darkorange')

# Вертикальна лінія — початок війни
ax.axvline(x=2022, color='red',
           linestyle='--',
           linewidth=2,
           label='Початок повномасштабної війни (лют. 2022)')

# Зони
ax.axvspan(2019, 2021.5, alpha=0.07, color='green', label='До війни')
ax.axvspan(2021.5, 2024, alpha=0.07, color='red', label='Під час війни')

# Оформлення
ax.set_xlabel('Рік', fontsize=12)
ax.set_ylabel('Середній NDVI (квітень–червень)', fontsize=12)
ax.set_title('Динаміка NDVI сільськогосподарських культур\nЖитомирська vs Херсонська область (2019–2024)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0.3, 0.7)
ax.grid(True, alpha=0.3)
ax.set_xticks(роки)

plt.tight_layout()
plt.savefig('ndvi_zhytomyr_kherson.png', dpi=300, bbox_inches='tight')
plt.show()

print("Графік збережено!")

In [ ]:
# 15: Офіційні дані Держстату
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

роки = [2019, 2020, 2021, 2022, 2023]

# Урожайність зернових (ц/га) — офіційний Держстат
zhytomyr_yield = [58.9, 48.2, 60.8, 43.3, 54.8]
kherson_yield  = [36.2, 35.0, 43.4, 31.9, None]

# NDVI з наших розрахунків GEE
zhytomyr_ndvi = [0.5424, 0.4579, 0.5318, 0.5610, 0.5809]
kherson_ndvi  = [0.3974, 0.3933, 0.4144, 0.3898, 0.4336]

# Створюємо таблицю
df = pd.DataFrame({
    'Рік': роки,
    'Житомирська_урожайність_цга': zhytomyr_yield,
    'Херсонська_урожайність_цга':  kherson_yield,
    'Житомирська_NDVI': zhytomyr_ndvi,
    'Херсонська_NDVI':  kherson_ndvi,
})

print(df.to_string(index=False))


In [ ]:
# 16: Головний рисунок статті
# Два показники на одному графіку — урожайність і NDVI

fig, ax1 = plt.subplots(figsize=(12, 7))

роки = [2019, 2020, 2021, 2022, 2023]
zhytomyr_yield = [58.9, 48.2, 60.8, 43.3, 54.8]
kherson_yield  = [36.2, 35.0, 43.4, 31.9, None]
zhytomyr_ndvi  = [0.5424, 0.4579, 0.5318, 0.5610, 0.5809]
kherson_ndvi   = [0.3974, 0.3933, 0.4144, 0.3898, 0.4336]

# Ліва вісь — урожайність
ax1.bar([р - 0.2 for р in роки], zhytomyr_yield,
        width=0.35, alpha=0.6,
        color='darkgreen', label='Житомирська — урожайність (ц/га)')
ax1.bar([р + 0.2 for р in роки],
        [v if v else 0 for v in kherson_yield],
        width=0.35, alpha=0.6,
        color='darkorange', label='Херсонська — урожайність (ц/га)')
ax1.set_ylabel('Урожайність зернових (ц/га)', fontsize=12, color='black')
ax1.set_ylim(0, 80)

# Права вісь — NDVI
ax2 = ax1.twinx()
ax2.plot(роки, zhytomyr_ndvi,
         color='green', linewidth=2.5,
         marker='o', markersize=8,
         label='Житомирська — NDVI')
ax2.plot(роки, kherson_ndvi,
         color='orange', linewidth=2.5,
         marker='s', markersize=8,
         linestyle='--',
         label='Херсонська — NDVI')
ax2.set_ylabel('Середній NDVI (квітень–червень)', fontsize=12)
ax2.set_ylim(0.3, 0.7)

# Лінія початку війни
ax1.axvline(x=2021.5, color='red',
            linestyle='--', linewidth=2.5,
            label='Початок повномасштабної війни')

# Позначка відсутніх даних
ax1.annotate('Дані відсутні\n(окупація)',
             xy=(2023, 5),
             ha='center', fontsize=9,
             color='red',
             bbox=dict(boxstyle='round', facecolor='lightyellow'))

# Легенда
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           loc='upper left', fontsize=9)

ax1.set_xlabel('Рік', fontsize=12)
ax1.set_title('Динаміка урожайності зернових та індексу NDVI\nЖитомирська та Херсонська області (2019–2023)',
              fontsize=14, fontweight='bold')
ax1.set_xticks(роки)
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('main_figure.png', dpi=300, bbox_inches='tight')
plt.show()

print("Головний рисунок статті збережено!")